<a href="https://colab.research.google.com/github/Ramprashanth17/Gen_AI/blob/main/rag-learning-platform/notebooks/4_RAG_Pipeline_Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Tutorial 04: Building the Complete RAG Pipeline

**Learning Objectives:**
- Connect all RAG components into a working system
- Master prompt engineering for RAG applications
- Compare RAG vs non-RAG answers to see the difference
- Implement error handling and edge cases
- Evaluate RAG system performance
- Build a production-ready RAG query function

**Prerequisites:**
- ✅ Completed `01_embeddings_fundamentals.ipynb`
- ✅ Completed `02_chunking_and_tokenization.ipynb`
- ✅ Completed `03_vector_storage_chromadb.ipynb`
- ✅ Understand embeddings, chunking, vector storage, and cosine similarity

**Time Required:** 50-60 minutes

---

## 📖 Table of Contents
1. [The RAG Promise: Why Bother?](#problem)
2. [Building the RAG Query Function](#building)
3. [Prompt Engineering for RAG](#prompting)
4. [The WOW Moment: RAG vs Baseline](#comparison)
5. [Advanced: Multi-Query RAG](#advanced)
6. [Error Handling & Edge Cases](#errors)
7. [Evaluation Metrics](#evaluation)
8. [Common Doubts & Pitfalls](#doubts)
9. [Practice Exercises](#exercises)

---

**Author:** Ramprashanth | INFO 7390 Final Project | December 2025

**🔗 Previous Tutorials:**
- [1_Embeddings_Fundamentals.ipynb](https://github.com/Ramprashanth17/Gen_AI/blob/main/rag-learning-platform/notebooks/1_Embeddings_Fundamentals.ipynb)
- [2_Chunking_and_tokenization.ipynb](https://github.com/Ramprashanth17/Gen_AI/blob/main/rag-learning-platform/notebooks/2_Chunking_and_Tokenization.ipynb)
- [3_Vector_storage_chromadb.ipynb](https://github.com/Ramprashanth17/Gen_AI/blob/main/rag-learning-platform/notebooks/3_Vector_Storage_Chromadb.ipynb)

**⏭️ Next:**
- `DEMO_full_rag_system.ipynb` - Everything in one streamlined demo!

### Setup and Getting Started!

In [ ]:
# Install packages
!pip install -q chromadb sentence-transformers google-generativeai tiktoken matplotlib

In [ ]:
# Imports
import chromadb
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from google.colab import userdata
import tiktoken
import time
from typing import List, Dict, Tuple
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Configure Gemini
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    print("✅ Gemini API configured")
except Exception as e:
    print(f"⚠️ API key error: {e}")
    print("Please set GEMINI_API_KEY in Colab Secrets")

# Initialize models
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
llm_model = genai.GenerativeModel('gemini-2.5-flash')

print("✅ Setup complete!")
print(f"📊 Embedding model: all-MiniLM-L6-v2 (384 dimensions)")
print(f"🤖 LLM model: Gemini 2.5 Flash")

<a id="problem"></a>
## 🤔 The RAG Promise: Why Bother With All This?

### The LLM Hallucination Problem

**Scenario:** You ask an LLM about your company's specific policies.
```python
# Without RAG - Just asking Gemini directly
query = "What is our company's remote work policy?"

response = llm.generate_content(query)
print(response.text)

# Possible output:
# "Most companies allow 2-3 days remote work per week..."
# ⚠️ GENERIC! Not YOUR company's actual policy!
# ⚠️ Could be completely wrong!
# ⚠️ No sources cited!
```

**Problems:**
- ❌ **Hallucination:** LLM makes up plausible-sounding but wrong answers
- ❌ **No specificity:** Generic knowledge, not YOUR data
- ❌ **No citations:** Can't verify the information
- ❌ **Outdated:** Training data might be old

---

### The RAG Solution

**Same question, but with RAG:**
```python
# With RAG - Provide relevant context first
query = "What is our company's remote work policy?"

# 1. Find relevant chunks from YOUR documents
relevant_chunks = vector_db.search(query, top_k=3)
# Returns: Actual policy text from employee_handbook.pdf

# 2. Give LLM the context
prompt = f"""
Context from company handbook:
{relevant_chunks}

Question: {query}

Answer based ONLY on the provided context:
"""

# 3. Generate grounded answer
response = llm.generate_content(prompt)

# Output:
# "According to the employee handbook (page 23), employees may
#  work remotely up to 3 days per week with manager approval..."
# ✅ SPECIFIC to your company!
# ✅ CITED source (page 23)!
# ✅ ACCURATE (from actual document)!
```

---

### The RAG Architecture (What You Built!)
```
USER QUESTION: "What is the remote work policy?"
        ↓
    ┌───────────────────────────────────────┐
    │  1. EMBED QUERY (Tutorial 01)         │
    │     "What is..." → [0.23, 0.89, ...]  │
    └───────────────────────────────────────┘
        ↓
    ┌───────────────────────────────────────┐
    │  2. VECTOR SEARCH (Tutorial 03)       │
    │     Find similar chunks in ChromaDB   │
    │     Top 3: [Chunk 15, Chunk 23, ...]  │
    └───────────────────────────────────────┘
        ↓
    ┌───────────────────────────────────────┐
    │  3. BUILD CONTEXT (Tutorial 02)       │
    │     Combine chunks with metadata      │
    │     Context = Chunk 15 + 23 + 31      │
    └───────────────────────────────────────┘
        ↓
    ┌───────────────────────────────────────┐
    │  4. PROMPT ENGINEERING (Tutorial 04!) │
    │     Create effective prompt for LLM   │
    └───────────────────────────────────────┘
        ↓
    ┌───────────────────────────────────────┐
    │  5. GENERATE ANSWER (Tutorial 04!)    │
    │     LLM creates grounded response     │
    │     WITH source citations!            │
    └───────────────────────────────────────┘
        ↓
    ACCURATE, CITED ANSWER ✅
```

**This is what we're building today!** 🎯

In [ ]:
# ═══════════════════════════════════════════════════════
# 📚 STEP 1: Create Our Knowledge Base
# ═══════════════════════════════════════════════════════

print("📚 BUILDING KNOWLEDGE BASE")
print("="*70)

# Sample documents (simulating a dog care guide)
knowledge_base = [
    {
        "text": "Dogs need at least 30-60 minutes of daily exercise depending on breed and age. High-energy breeds like Border Collies and Australian Shepherds may need 2+ hours. Regular exercise prevents obesity, reduces anxiety, and promotes better behavior. Activities can include walks, runs, fetch, swimming, or agility training.",
        "metadata": {
            "source": "dog_care_guide.pdf",
            "page": 12,
            "section": "Exercise Requirements",
            "topic": "exercise"
        }
    },
    {
        "text": "A balanced dog diet should contain high-quality protein (chicken, beef, fish), healthy fats (omega-3, omega-6), carbohydrates (rice, sweet potato), and essential vitamins and minerals. Feed adult dogs twice daily. Portion sizes depend on weight, age, and activity level. Avoid chocolate, grapes, onions, and xylitol - these are toxic to dogs.",
        "metadata": {
            "source": "dog_care_guide.pdf",
            "page": 18,
            "section": "Nutrition Guidelines",
            "topic": "nutrition"
        }
    },
    {
        "text": "Puppy training should begin at 8-10 weeks old. Start with basic commands: sit, stay, come, down, and leave it. Use positive reinforcement - reward desired behaviors with treats, praise, or play. Training sessions should be 10-15 minutes, 2-3 times daily. Consistency is crucial. Never use physical punishment as it damages trust and can create fear-based behaviors.",
        "metadata": {
            "source": "training_manual.pdf",
            "page": 5,
            "section": "Basic Training",
            "topic": "training"
        }
    },
    {
        "text": "Schedule annual vet checkups for adult dogs. Puppies need visits every 3-4 weeks until 16 weeks old for vaccinations. Senior dogs (7+ years) should see the vet twice annually. Watch for warning signs: lethargy, loss of appetite, excessive thirst, vomiting, diarrhea, or behavioral changes. Early detection saves lives and reduces treatment costs.",
        "metadata": {
            "source": "health_guide.pdf",
            "page": 8,
            "section": "Veterinary Care",
            "topic": "health"
        }
    },
    {
        "text": "Common dog health issues include dental disease (affects 80% of dogs by age 3), obesity (affects 56% of dogs), arthritis in senior dogs, ear infections, and skin allergies. Regular dental cleaning, weight management, and preventive care reduce these risks. Pet insurance can help manage unexpected veterinary costs.",
        "metadata": {
            "source": "health_guide.pdf",
            "page": 15,
            "section": "Common Health Issues",
            "topic": "health"
        }
    },
    {
        "text": "Grooming needs vary by breed. Long-haired breeds need daily brushing to prevent matting. Short-haired breeds can be brushed weekly. Bathe dogs every 4-8 weeks or as needed. Trim nails monthly. Clean ears weekly to prevent infections. Regular grooming strengthens your bond and lets you check for skin issues or parasites.",
        "metadata": {
            "source": "grooming_guide.pdf",
            "page": 3,
            "section": "Grooming Basics",
            "topic": "grooming"
        }
    }
]

print(f"✅ Created knowledge base with {len(knowledge_base)} documents")
print(f"📊 Topics covered: exercise, nutrition, training, health, grooming")

# Display preview
print("\n📋 KNOWLEDGE BASE PREVIEW:")
print("-"*70)
for i, doc in enumerate(knowledge_base[:3], 1):
    print(f"\nDocument {i}:")
    print(f"  Topic: {doc['metadata']['topic']}")
    print(f"  Source: {doc['metadata']['source']}, Page {doc['metadata']['page']}")
    print(f"  Text: {doc['text'][:80]}...")



In [ ]:
# ═══════════════════════════════════════════════════════
# 📚 STEP 2: Store in ChromaDB
# ═══════════════════════════════════════════════════════

print("\n" + "="*70)
print("💾 STORING IN CHROMADB")
print("="*70)

# Create ChromaDB client and collection
client = chromadb.Client()
collection = client.create_collection(
    name="dog_care_kb",
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity!
)

# Generate embeddings
texts = [doc['text'] for doc in knowledge_base]
embeddings = embedding_model.encode(texts)

print(f"✅ Generated {len(embeddings)} embeddings")
print(f"   Shape: {embeddings.shape}")

# Store in ChromaDB
collection.add(
    ids=[f"doc_{i}" for i in range(len(knowledge_base))],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[doc['metadata'] for doc in knowledge_base]
)

print(f"✅ Stored {collection.count()} documents in ChromaDB")
print("\n🎯 Knowledge base ready for RAG queries! 🚀")

<a id="building"></a>
## 🔨 Building the RAG Query Function

Now let's build the **core RAG function** that brings everything together!

### The RAG Pipeline (Step by Step)
```
Query → Embed → Search → Retrieve → Build Prompt → Generate → Return
```

Let's implement each step:

In [ ]:
# ═══════════════════════════════════════════════════════
# 🔨 BUILDING THE RAG QUERY FUNCTION (CORRECTED)
# ═══════════════════════════════════════════════════════

def rag_query(question: str, collection, llm_model, top_k: int = 3, verbose: bool = True) -> Dict:
    """
    Complete RAG pipeline: Retrieve relevant context and generate answer.

    Args:
        question: User's question
        collection: ChromaDB collection with documents
        llm_model: Gemini model for generation
        top_k: Number of chunks to retrieve
        verbose: Print step-by-step process

    Returns:
        Dict with answer, sources, context, and metrics
    """

    if verbose:
        print("🔄 RAG PIPELINE EXECUTING...")
        print("="*70)

    # ═══════════════════════════════════════════════════════
    # STEP 1: Retrieve Relevant Chunks
    # ═══════════════════════════════════════════════════════

    start_time = time.time()

    if verbose:
        print(f"\n1️⃣ RETRIEVING relevant chunks for: '{question}'")

    # Search vector database
    results = collection.query(
        query_texts=[question],
        n_results=top_k
    )

    # Extract results
    retrieved_docs = results['documents'][0]
    retrieved_metadata = results['metadatas'][0]
    distances = results['distances'][0]
    similarities = [1 - d for d in distances]

    retrieval_time = time.time() - start_time

    if verbose:
        print(f"   ✅ Retrieved {len(retrieved_docs)} chunks in {retrieval_time:.3f}s")
        for i, (doc, meta, sim) in enumerate(zip(retrieved_docs, retrieved_metadata, similarities), 1):
            print(f"   {i}. {meta['section']} (similarity: {sim:.3f})")

    # ═══════════════════════════════════════════════════════
    # STEP 2: Build Context from Retrieved Chunks
    # ═══════════════════════════════════════════════════════

    if verbose:
        print(f"\n2️⃣ BUILDING context from retrieved chunks")

    # Format context with sources
    context_parts = []
    for i, (doc, meta) in enumerate(zip(retrieved_docs, retrieved_metadata), 1):
        context_parts.append(
            f"[Source {i}: {meta['source']}, Page {meta['page']}, Section: {meta['section']}]\n{doc}"
        )

    context = "\n\n".join(context_parts)

    # Calculate context tokens (MOVED OUTSIDE verbose block - BUG FIX!)
    tokenizer = tiktoken.get_encoding("cl100k_base")
    context_tokens = len(tokenizer.encode(context))

    if verbose:
        print(f"   ✅ Context built: {len(context)} characters")
        print(f"   📊 Context tokens: {context_tokens}")

    # ═══════════════════════════════════════════════════════
    # STEP 3: Create RAG Prompt
    # ═══════════════════════════════════════════════════════

    if verbose:
        print(f"\n3️⃣ CREATING RAG prompt")

    prompt = f"""You are a helpful assistant answering questions about dog care.

Context Information (from trusted sources):
{context}

User Question: {question}

Instructions:
- Answer the question using ONLY the information provided in the context above
- If the context doesn't contain enough information, say so clearly
- Cite your sources using the [Source X] references provided
- Be concise but complete (2-4 sentences)
- Provide actionable advice when appropriate

Answer:"""

    if verbose:
        prompt_tokens = len(tokenizer.encode(prompt))
        print(f"   ✅ Prompt created: {prompt_tokens} tokens total")

    # ═══════════════════════════════════════════════════════
    # STEP 4: Generate Answer with LLM
    # ═══════════════════════════════════════════════════════

    if verbose:
        print(f"\n4️⃣ GENERATING answer with Gemini")

    generation_start = time.time()

    try:
        response = llm_model.generate_content(prompt)
        answer = response.text
        generation_time = time.time() - generation_start

        if verbose:
            print(f"   ✅ Answer generated in {generation_time:.3f}s")
            response_tokens = len(tokenizer.encode(answer))
            print(f"   📊 Response tokens: {response_tokens}")

    except Exception as e:
        answer = f"Error generating response: {str(e)}"
        generation_time = 0
        if verbose:
            print(f"   ❌ Error: {e}")

    # ═══════════════════════════════════════════════════════
    # STEP 5: Package Results
    # ═══════════════════════════════════════════════════════

    total_time = time.time() - start_time

    result = {
        'question': question,
        'answer': answer,
        'sources': [
            {
                'text': doc,
                'source': meta['source'],
                'page': meta['page'],
                'section': meta['section'],
                'similarity': sim
            }
            for doc, meta, sim in zip(retrieved_docs, retrieved_metadata, similarities)
        ],
        'context': context,
        'metrics': {
            'retrieval_time': retrieval_time,
            'generation_time': generation_time,
            'total_time': total_time,
            'context_tokens': context_tokens,  # Now always defined!
            'chunks_retrieved': len(retrieved_docs),
            'avg_similarity': np.mean(similarities)
        }
    }

    if verbose:
        print(f"\n" + "="*70)
        print(f"✅ RAG QUERY COMPLETE in {total_time:.3f}s")
        print("="*70)

    return result


In [ ]:
# ═══════════════════════════════════════════════════════
# 🧪 TEST THE RAG FUNCTION
# ═══════════════════════════════════════════════════════

print("\n🧪 TESTING RAG QUERY FUNCTION")
print("="*70)

test_question = "How much exercise does a dog need?"

result = rag_query(
    question=test_question,
    collection=collection,
    llm_model=llm_model,
    top_k=3,
    verbose=True
)

print("\n" + "="*70)
print("📝 FINAL ANSWER:")
print("="*70)
print(result['answer'])

print("\n" + "="*70)
print("📚 SOURCES USED:")
print("="*70)
for i, source in enumerate(result['sources'], 1):
    print(f"\n{i}. {source['source']}, Page {source['page']}")
    print(f"   Section: {source['section']}")
    print(f"   Similarity: {source['similarity']:.3f}")
    print(f"   Text: {source['text'][:100]}...")

<a id="prompting"></a>
## 🎨 Prompt Engineering for RAG

**The prompt is CRITICAL!** A bad prompt wastes good retrieval.

### Anatomy of a Good RAG Prompt
```python
# ❌ BAD RAG PROMPT (Too simple)
prompt = f"{context}\n\nQuestion: {question}"
# Problems:
# - No instructions to stay grounded
# - No citation guidance
# - LLM might ignore context and hallucinate anyway

# ✅ GOOD RAG PROMPT (Structured)
prompt = f"""
You are a helpful assistant.

Context (from verified sources):
{context}

Question: {question}

Instructions:
- Answer using ONLY the context provided
- Cite sources using [Source X] format
- If context insufficient, say "I don't have enough information"
- Be concise but complete

Answer:
"""
```

---

### Key Prompt Engineering Principles

**1. Set Clear Role**
```python
"You are an expert dog care advisor..." ✅
vs
"You are an AI..." ❌ (too generic)
```

**2. Emphasize Grounding**
```python
"Answer using ONLY the context provided" ✅
"If the context doesn't have the info, say so" ✅
```

**3. Request Citations**
```python
"Cite sources using [Source X] format" ✅
"Include page numbers when relevant" ✅
```

**4. Specify Format**
```python
"Provide a concise answer in 2-3 sentences" ✅
"List key points with bullet points" ✅
```

**5. Handle Uncertainty**
```python
"If unsure, say 'Based on the available information...'" ✅
"Never make up information" ✅
```

In [ ]:
# ═══════════════════════════════════════════════════════
# 🔬 EXPERIMENT: Different Prompt Styles
# ═══════════════════════════════════════════════════════

print("🔬 PROMPT ENGINEERING EXPERIMENTS")
print("="*70)

question = "What should I feed my dog?"

# Get context (same for all prompts)
results = collection.query(query_texts=[question], n_results=2)
context = "\n\n".join(results['documents'][0])

# ───────────────────────────────────────────────────────
# Prompt Style 1: Minimal (Baseline)
# ───────────────────────────────────────────────────────

prompt_minimal = f"{context}\n\nQuestion: {question}\n\nAnswer:"

print("\n📝 STYLE 1: Minimal Prompt")
print("-"*70)
print("Prompt:", prompt_minimal[:100] + "...")

response1 = llm_model.generate_content(prompt_minimal)
print(f"\nResponse: {response1.text}")





In [ ]:
# ───────────────────────────────────────────────────────
# Prompt Style 2: Structured (Recommended)
# ───────────────────────────────────────────────────────

prompt_structured = f"""You are a helpful dog care expert.

Context from verified sources:
{context}

Question: {question}

Instructions:
- Answer using ONLY the context above
- Cite specific sources when possible
- Be concise (2-3 sentences)

Answer:"""

print("\n" + "="*70)
print("📝 STYLE 2: Structured Prompt")
print("-"*70)

response2 = llm_model.generate_content(prompt_structured)
print(f"\nResponse: {response2.text}")

In [ ]:
# ───────────────────────────────────────────────────────
# Prompt Style 3: Detailed Instructions (Best)
# ───────────────────────────────────────────────────────

prompt_detailed = f"""You are an expert dog care advisor helping pet owners.

CONTEXT (from trusted veterinary sources):
{context}

USER QUESTION: {question}

INSTRUCTIONS:
1. Answer the question using ONLY the information in the context above
2. If the context doesn't contain relevant information, clearly state: "I don't have specific information about that in the provided sources"
3. Cite your sources by mentioning the relevant information
4. Provide practical, actionable advice
5. Keep your answer concise (3-4 sentences maximum)
6. Do not add information not present in the context

YOUR ANSWER:"""

print("\n" + "="*70)
print("📝 STYLE 3: Detailed Instructions")
print("-"*70)

response3 = llm_model.generate_content(prompt_detailed)
print(f"\nResponse: {response3.text}")

print("\n" + "="*70)
print("🎯 COMPARISON:")
print("Style 1: Basic, may hallucinate")
print("Style 2: Better, includes citations")
print("Style 3: Best, grounded and actionable ✅")
print("\nRecommendation: Use Style 3 for production!")
print("="*70)

<a id="comparison"></a>
## ✨ The WOW Moment: RAG vs Non-RAG Comparison

**This is where you see the MAGIC of RAG!**

Let's ask questions that the LLM wouldn't know without RAG.

In [ ]:
# ═══════════════════════════════════════════════════════
# ✨ RAG vs Baseline Comparison (RATE-LIMIT SAFE)
# ═══════════════════════════════════════════════════════

def compare_rag_vs_baseline(question: str, collection, llm_model, top_k: int = 3,
                            delay_between_calls: float = 5.0):
    """
    Compare RAG answer vs baseline LLM answer side-by-side.

    RATE LIMIT SAFE: Adds delays between API calls
    """
    print("="*70)
    print(f"🔍 QUESTION: {question}")
    print("="*70)

    # ───────────────────────────────────────────────────────
    # BASELINE: Ask LLM directly (no context)
    # ───────────────────────────────────────────────────────

    print("\n❌ WITHOUT RAG (Baseline LLM):")
    print("-"*70)

    baseline_start = time.time()

    try:
        baseline_response = llm_model.generate_content(question)
        baseline_answer = baseline_response.text
        baseline_time = time.time() - baseline_start

        print(baseline_answer)
        print(f"\n⏱️  Time: {baseline_time:.3f}s")
        print("⚠️  Issues: Generic, no sources, might be inaccurate")

    except Exception as e:
        if "429" in str(e) or "quota" in str(e).lower():
            print("⚠️ RATE LIMIT HIT - Skipping baseline for this query")
            print("💡 In production, implement exponential backoff retry")
            baseline_answer = "[Rate limited - skipped]"
            baseline_time = 0
        else:
            print(f"❌ Error: {e}")
            baseline_answer = f"Error: {e}"
            baseline_time = 0

    # ⏸️ RATE LIMIT PROTECTION: Wait before next call
    print(f"\n⏸️  Waiting {delay_between_calls}s to avoid rate limits...")
    time.sleep(delay_between_calls)

    # ───────────────────────────────────────────────────────
    # RAG: Retrieve context first, then answer
    # ───────────────────────────────────────────────────────

    print("\n" + "="*70)
    print("✅ WITH RAG (Context-Enhanced):")
    print("-"*70)

    try:
        rag_result = rag_query(
            question=question,
            collection=collection,
            llm_model=llm_model,
            top_k=top_k,
            verbose=False
        )

        print(rag_result['answer'])
        print(f"\n📚 Sources:")
        for i, source in enumerate(rag_result['sources'], 1):
            print(f"   {i}. {source['source']}, Page {source['page']} (similarity: {source['similarity']:.3f})")
        print(f"\n⏱️  Time: {rag_result['metrics']['total_time']:.3f}s")
        print(f"📊 Retrieved: {rag_result['metrics']['chunks_retrieved']} chunks, {rag_result['metrics']['context_tokens']} tokens")

    except Exception as e:
        if "429" in str(e) or "quota" in str(e).lower():
            print("⚠️ RATE LIMIT HIT")
            print(f"💡 Please wait 60 seconds and try again")
            rag_result = None
        else:
            print(f"❌ Error: {e}")
            rag_result = None

    # ───────────────────────────────────────────────────────
    # Side-by-side comparison (if both succeeded)
    # ───────────────────────────────────────────────────────

    if rag_result and baseline_answer != "[Rate limited - skipped]":
        print("\n" + "="*70)
        print("📊 COMPARISON SUMMARY:")
        print("="*70)

        comparison_df = pd.DataFrame({
            'Metric': [
                'Response Quality',
                'Sources Cited',
                'Specific to Query',
                'Verifiable',
                'Processing Time',
                'Cost (estimate)'
            ],
            'Baseline (No RAG)': [
                '⭐⭐⭐ Generic',
                '❌ None',
                '⚠️ General knowledge',
                '❌ No',
                f'{baseline_time:.3f}s',
                '~$0.00001'
            ],
            'RAG-Enhanced': [
                '⭐⭐⭐⭐⭐ Precise',
                f"✅ {len(rag_result['sources'])} sources",
                '✅ Domain-specific',
                '✅ Yes (with pages)',
                f"{rag_result['metrics']['total_time']:.3f}s",
                f"~$0.0003 ({rag_result['metrics']['context_tokens']} tokens)"
            ]
        })

        print(comparison_df.to_string(index=False))

        print("\n🎯 VERDICT:")
        print("RAG costs 30x more but provides:")
        print("  ✅ Accurate, specific answers")
        print("  ✅ Verifiable sources")
        print("  ✅ Domain expertise")
        print("  ✅ Up-to-date information")
        print("\nFor critical applications: RAG is worth it! 💯")

    # ⏸️ Wait before returning
    time.sleep(delay_between_calls)

    return {
        'baseline': baseline_answer if 'baseline_answer' in locals() else None,
        'rag': rag_result['answer'] if rag_result else None,
        'baseline_time': baseline_time if 'baseline_time' in locals() else 0,
        'rag_time': rag_result['metrics']['total_time'] if rag_result else 0
    }


# ═══════════════════════════════════════════════════════
# 🧪 TEST: With Proper Rate Limiting
# ═══════════════════════════════════════════════════════

print("\n\n🧪 TESTING WITH RATE LIMIT PROTECTION")
print("="*70)
print("⚠️ Note: Each comparison takes ~10 seconds due to API limits")
print("="*70)

test_questions = [
    "How much exercise does a dog need daily?",
    "What foods are toxic to dogs?",
    # "When should I start training my puppy?"  # Comment out to save API calls
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'🔹'*35}")
    print(f"Test {i}/{len(test_questions)}")
    print(f"{'🔹'*35}")

    compare_rag_vs_baseline(
        question,
        collection,
        llm_model,
        top_k=2,
        delay_between_calls=12  # 12 second delay (safe for free tier)
    )

print("\n" + "="*70)
print("✅ COMPARISON COMPLETE!")
print("="*70)

## ⚠️ Important: API Rate Limits

### What Just Happened?

If you saw a `429 TooManyRequests` error, you hit Gemini's rate limit!

**Gemini Free Tier Limits:**
- 5 requests per minute
- 1,500 requests per day

**Why This Matters for RAG:**
```python
# Each RAG query = 1 API call
# Each baseline query = 1 API call
# Comparison = 2 API calls per question

# Testing 3 questions:
3 questions × 2 calls = 6 API calls
But limit = 5 calls/minute
→ Error on 6th call! ⚠️
```

---

### Solutions:

**1. Add Delays (Simple)**
```python
time.sleep(12)  # Wait 12 seconds between queries
# Stays under 5 per minute
```

**2. Batch Processing (Better)**
```python
# Test in batches
questions_batch_1 = questions[:2]  # First 2
time.sleep(60)  # Wait 1 minute
questions_batch_2 = questions[2:4]  # Next 2
```

**3. Exponential Backoff (Production)**
```python
# Automatically retry with increasing delays
# 5s → 10s → 20s → 40s
```

**4. Upgrade API Plan (For production)**
- Gemini paid tier: 1000+ RPM
- No rate limits for serious applications



### Query which could result in the above error

```
# ═══════════════════════════════════════════════════════
# ✨ THE WOW MOMENT: RAG vs Baseline Comparison
# ═══════════════════════════════════════════════════════

def compare_rag_vs_baseline(question: str, collection, llm_model, top_k: int = 3):
    """
    Compare RAG answer vs baseline LLM answer side-by-side.
    """
    print("="*70)
    print(f"🔍 QUESTION: {question}")
    print("="*70)
    
    # ───────────────────────────────────────────────────────
    # BASELINE: Ask LLM directly (no context)
    # ───────────────────────────────────────────────────────
    
    print("\n❌ WITHOUT RAG (Baseline LLM):")
    print("-"*70)
    
    baseline_start = time.time()
    baseline_response = llm_model.generate_content(question)
    baseline_time = time.time() - baseline_start
    
    print(baseline_response.text)
    print(f"\n⏱️  Time: {baseline_time:.3f}s")
    print("⚠️  Issues: Generic, no sources, might be inaccurate")
    
    # ───────────────────────────────────────────────────────
    # RAG: Retrieve context first, then answer
    # ───────────────────────────────────────────────────────
    
    print("\n" + "="*70)
    print("✅ WITH RAG (Context-Enhanced):")
    print("-"*70)
    
    rag_result = rag_query(
        question=question,
        collection=collection,
        llm_model=llm_model,
        top_k=top_k,
        verbose=False  # Quiet mode
    )
    
    print(rag_result['answer'])
    print(f"\n📚 Sources:")
    for i, source in enumerate(rag_result['sources'], 1):
        print(f"   {i}. {source['source']}, Page {source['page']} (similarity: {source['similarity']:.3f})")
    print(f"\n⏱️  Time: {rag_result['metrics']['total_time']:.3f}s")
    print(f"📊 Retrieved: {rag_result['metrics']['chunks_retrieved']} chunks, {rag_result['metrics']['context_tokens']} tokens")
    
    # ───────────────────────────────────────────────────────
    # Side-by-side comparison
    # ───────────────────────────────────────────────────────
    
    print("\n" + "="*70)
    print("📊 COMPARISON SUMMARY:")
    print("="*70)
    
    comparison_df = pd.DataFrame({
        'Metric': [
            'Response Quality',
            'Sources Cited',
            'Specific to Query',
            'Verifiable',
            'Processing Time',
            'Cost (estimate)'
        ],
        'Baseline (No RAG)': [
            '⭐⭐⭐ Generic',
            '❌ None',
            '⚠️ General knowledge',
            '❌ No',
            f'{baseline_time:.3f}s',
            '~$0.00001'
        ],
        'RAG-Enhanced': [
            '⭐⭐⭐⭐⭐ Precise',
            f"✅ {len(rag_result['sources'])} sources",
            '✅ Domain-specific',
            '✅ Yes (with pages)',
            f"{rag_result['metrics']['total_time']:.3f}s",
            f"~$0.0003 ({rag_result['metrics']['context_tokens']} tokens)"
        ]
    })
    
    print(comparison_df.to_string(index=False))
    
    print("\n🎯 VERDICT:")
    print("RAG costs 30x more but provides:")
    print("  ✅ Accurate, specific answers")
    print("  ✅ Verifiable sources")
    print("  ✅ Domain expertise")
    print("  ✅ Up-to-date information")
    print("\nFor critical applications: RAG is worth it! 💯")
    
    return {
        'baseline': baseline_response.text,
        'rag': rag_result['answer'],
        'baseline_time': baseline_time,
        'rag_time': rag_result['metrics']['total_time']
    }


# ═══════════════════════════════════════════════════════
# 🧪 TEST: Multiple Questions
# ═══════════════════════════════════════════════════════

print("\n\n🧪 TESTING WITH DIFFERENT QUESTIONS")
print("="*70)

test_questions = [
    "How much exercise does a dog need daily?",
    "What foods are toxic to dogs?",
    "When should I start training my puppy?"
]

for question in test_questions:
    print("\n" + "🔹"*35)
    compare_rag_vs_baseline(question, collection, llm_model, top_k=2)
    print("🔹"*35)
    time.sleep(2)  # Rate limiting for API

```

In [ ]:
# ═══════════════════════════════════════════════════════
# 📊 VISUALIZING RAG PERFORMANCE
# ═══════════════════════════════════════════════════════

print("📊 RAG PERFORMANCE VISUALIZATION")
print("="*70)

# Test multiple queries and collect metrics
test_queries = [
    "How much exercise do dogs need?",
    "What should I feed my dog?",
    "How do I train my puppy?",
    "When should I visit the vet?",
    "How often should I groom my dog?"
]

metrics_data = []

for query in test_queries:
    result = rag_query(query, collection, llm_model, top_k=3, verbose=False)

    metrics_data.append({
        'Query': query[:30] + "...",
        'Retrieval Time': result['metrics']['retrieval_time'],
        'Generation Time': result['metrics']['generation_time'],
        'Total Time': result['metrics']['total_time'],
        'Chunks Retrieved': result['metrics']['chunks_retrieved'],
        'Avg Similarity': result['metrics']['avg_similarity'],
        'Context Tokens': result['metrics']['context_tokens']
    })

    time.sleep(1)  # Rate limiting

# Create DataFrame
df_metrics = pd.DataFrame(metrics_data)

# ═══════════════════════════════════════════════════════
# Visualization
# ═══════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Time breakdown
ax1 = axes[0, 0]
x = range(len(df_metrics))
width = 0.35

ax1.bar([i - width/2 for i in x], df_metrics['Retrieval Time'], width,
        label='Retrieval', alpha=0.8, color='#3498db')
ax1.bar([i + width/2 for i in x], df_metrics['Generation Time'], width,
        label='Generation', alpha=0.8, color='#e74c3c')

ax1.set_xlabel('Query')
ax1.set_ylabel('Time (seconds)')
ax1.set_title('RAG Pipeline: Time Breakdown', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([f"Q{i+1}" for i in x])
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Similarity scores
ax2 = axes[0, 1]
ax2.bar(x, df_metrics['Avg Similarity'], alpha=0.8, color='#2ecc71')
ax2.axhline(y=0.7, color='green', linestyle='--', alpha=0.5, label='High similarity')
ax2.axhline(y=0.4, color='orange', linestyle='--', alpha=0.5, label='Medium similarity')
ax2.set_xlabel('Query')
ax2.set_ylabel('Average Similarity Score')
ax2.set_title('Retrieval Quality: Similarity Scores', fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels([f"Q{i+1}" for i in x])
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Plot 3: Token usage
ax3 = axes[1, 0]
ax3.bar(x, df_metrics['Context Tokens'], alpha=0.8, color='#f39c12')
ax3.set_xlabel('Query')
ax3.set_ylabel('Tokens')
ax3.set_title('Context Size: Token Usage', fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels([f"Q{i+1}" for i in x])
ax3.grid(axis='y', alpha=0.3)

# Add cost labels
for i, tokens in enumerate(df_metrics['Context Tokens']):
    cost = tokens * 0.001 / 1000  # Rough estimate
    ax3.text(i, tokens + 5, f'${cost:.5f}', ha='center', fontsize=8)

# Plot 4: Summary metrics table
ax4 = axes[1, 1]
ax4.axis('off')

summary_data = [
    ['Avg Retrieval Time', f"{df_metrics['Retrieval Time'].mean():.3f}s"],
    ['Avg Generation Time', f"{df_metrics['Generation Time'].mean():.3f}s"],
    ['Avg Total Time', f"{df_metrics['Total Time'].mean():.3f}s"],
    ['Avg Similarity', f"{df_metrics['Avg Similarity'].mean():.3f}"],
    ['Avg Context Tokens', f"{df_metrics['Context Tokens'].mean():.0f}"],
    ['Avg Cost/Query', f"${(df_metrics['Context Tokens'].mean() * 0.001 / 1000):.5f}"]
]

table = ax4.table(
    cellText=summary_data,
    colLabels=['Metric', 'Value'],
    cellLoc='left',
    loc='center',
    colWidths=[0.6, 0.4]
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

# Style table
for i in range(2):
    table[(0, i)].set_facecolor('#3498db')
    table[(0, i)].set_text_props(weight='bold', color='white')

for i in range(1, len(summary_data) + 1):
    for j in range(2):
        table[(i, j)].set_facecolor('#ecf0f1' if i % 2 == 0 else 'white')

ax4.set_title('Performance Summary', fontweight='bold', fontsize=12, pad=20)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("🎯 INSIGHTS FROM VISUALIZATION:")
print("1. Retrieval is FAST (<0.1s typically)")
print("2. Generation takes longer (~1-2s)")
print("3. Similarity scores show retrieval quality")
print("4. Cost is proportional to context size")
print("="*70)

<a id="advanced"></a>
## 🚀 Advanced: Multi-Query RAG

**Problem:** Sometimes one query isn't enough to capture user intent.

**Solution:** Generate multiple variations of the query, retrieve for each, then combine results!

In [ ]:
# # ═══════════════════════════════════════════════════════
# # 🚀 ADVANCED: Multi-Query RAG
# # ═══════════════════════════════════════════════════════

# def multi_query_rag(question: str, collection, llm_model, num_variations: int = 3):
#     """
#     Generate multiple query variations for better retrieval coverage.
#     """

#     print(f"🚀 MULTI-QUERY RAG")
#     print("="*70)
#     print(f"Original question: '{question}'")

#     # ───────────────────────────────────────────────────────
#     # STEP 1: Generate query variations using LLM
#     # ───────────────────────────────────────────────────────

#     variation_prompt = f"""Generate {num_variations} different ways to ask this question:

# Original: "{question}"

# Requirements:
# - Keep the same meaning
# - Use different phrasing and keywords
# - Make them diverse

# Return ONLY the {num_variations} questions, one per line, no numbering."""

#     variation_response = llm_model.generate_content(variation_prompt)
#     variations = [q.strip() for q in variation_response.text.strip().split('\n') if q.strip()][:num_variations]

#     print(f"\n📝 Generated {len(variations)} query variations:")
#     for i, var in enumerate(variations, 1):
#         print(f"   {i}. {var}")

#     # ───────────────────────────────────────────────────────
#     # STEP 2: Retrieve for each variation
#     # ───────────────────────────────────────────────────────

#     print(f"\n🔍 Retrieving for each variation...")

#     all_chunks = []
#     all_similarities = []
#     seen_ids = set()

#     for var in variations:
#         results = collection.query(
#             query_texts=[var],
#             n_results=2  # Get top 2 for each variation
#         )

#         for doc, meta, dist in zip(
#             results['documents'][0],
#             results['metadatas'][0],
#             results['distances'][0]
#         ):
#             chunk_id = f"{meta['source']}_{meta['page']}_{meta['section']}"

#             # Deduplicate
#             if chunk_id not in seen_ids:
#                 seen_ids.add(chunk_id)
#                 all_chunks.append({
#                     'text': doc,
#                     'metadata': meta,
#                     'similarity': 1 - dist
#                 })
#                 all_similarities.append(1 - dist)

#     # Sort by similarity
#     all_chunks.sort(key=lambda x: x['similarity'], reverse=True)

#     # Take top K overall
#     top_chunks = all_chunks[:3]

#     print(f"   ✅ Retrieved {len(all_chunks)} unique chunks total")
#     print(f"   📊 Using top {len(top_chunks)} for context")

#     for i, chunk in enumerate(top_chunks, 1):
#         print(f"   {i}. {chunk['metadata']['section']} (sim: {chunk['similarity']:.3f})")

#     # ───────────────────────────────────────────────────────
#     # STEP 3: Generate answer with combined context
#     # ───────────────────────────────────────────────────────

#     context = "\n\n".join([
#         f"[Source: {c['metadata']['source']}, Page {c['metadata']['page']}]\n{c['text']}"
#         for c in top_chunks
#     ])

#     prompt = f"""Context from verified sources:
# {context}

# Question: {question}

# Answer using the context above, cite sources:"""

#     response = llm_model.generate_content(prompt)

#     print(f"\n💬 ANSWER:")
#     print("-"*70)
#     print(response.text)

#     return {
#         'answer': response.text,
#         'variations': variations,
#         'chunks_retrieved': len(all_chunks),
#         'sources': top_chunks
#     }

# ═══════════════════════════════════════════════════════
# 🧪 TEST: Multi-Query RAG
# ═══════════════════════════════════════════════════════

# result = multi_query_rag(
#     "What's the best way to keep my dog healthy?",
#     collection,
#     llm_model,
#     num_variations=3
# )

# print("\n" + "="*70)
# print("🎯 MULTI-QUERY BENEFIT:")
# print("By asking the question in multiple ways, we retrieve")
# print("MORE diverse, relevant chunks than a single query!")
# print("="*70)


""" Had to comment out as it is making too many API requests """

In [ ]:
# ═══════════════════════════════════════════════════════
# 🚀 MULTI-QUERY RAG DEMO (Using Example Results)
# ═══════════════════════════════════════════════════════

print("🚀 MULTI-QUERY RAG DEMONSTRATION")
print("="*70)
print("💡 Note: Using example results to avoid rate limits")
print("="*70)

# Show the concept
original_question = "What's the best way to keep my dog healthy?"

print(f"\n📌 Original Question: '{original_question}'")

# Example variations (what LLM would generate)
example_variations = [
    "What are the most effective strategies for promoting my dog's overall well-being?",
    "How can I best ensure my canine companion stays in peak health?",
    "What are the essential steps for maintaining my pet's long-term vitality?"
]

print(f"\n📝 LLM Generated {len(example_variations)} Variations:")
for i, var in enumerate(example_variations, 1):
    print(f"   {i}. {var}")

# Example retrieval results
print(f"\n🔍 Multi-Query Retrieval Results:")
print("-"*70)

example_results = [
    {
        'section': 'Veterinary Care',
        'similarity': 0.503,
        'source': 'health_guide.pdf',
        'page': 8
    },
    {
        'section': 'Nutrition Guidelines',
        'similarity': 0.468,
        'source': 'dog_care_guide.pdf',
        'page': 18
    },
    {
        'section': 'Exercise Requirements',
        'similarity': 0.455,
        'source': 'dog_care_guide.pdf',
        'page': 12
    }
]

for i, res in enumerate(example_results, 1):
    print(f"   {i}. {res['section']} from {res['source']} (similarity: {res['similarity']:.3f})")

# Example final answer
example_answer = """To keep your dog healthy, focus on three key areas:

1. **Regular Veterinary Care**: Schedule annual checkups for adult dogs and more frequent visits for puppies and seniors. Keep vaccinations up to date [Source 1: health_guide.pdf, Page 8].

2. **Proper Nutrition**: Provide a balanced diet with high-quality protein, healthy fats, and appropriate portions based on weight and activity level [Source 2: dog_care_guide.pdf, Page 18].

3. **Daily Exercise**: Ensure your dog gets 30-60 minutes of physical activity daily, adjusted for breed and age [Source 3: dog_care_guide.pdf, Page 12].

By addressing all three areas consistently, you'll maximize your dog's health and longevity."""

print(f"\n💬 Generated Answer:")
print("-"*70)
print(example_answer)

print("\n" + "="*70)
print("🎯 MULTI-QUERY BENEFIT:")
print("By rephrasing the question 3 different ways, we retrieved:")
print("  ✅ Health care information (vet visits)")
print("  ✅ Nutrition guidance (diet)")
print("  ✅ Exercise requirements (activity)")
print("\nA single query might have missed one of these important aspects!")
print("="*70)

print("\n💡 TO RUN LIVE:")
print("Wait 60 seconds, then uncomment and run:")
print("# result = multi_query_rag(question, collection, llm_model)")